In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://debales.ai"
res = requests.get(url)
soup = BeautifulSoup(res.text, "html.parser")

text = soup.get_text()

In [4]:
text

"Debales AI - Autonomous Logistics Automation Platform | AI Agents for Supply Chain | Debales AISolutions IntegrationsAI AgentsBlogCase StudiesSign UpSign InBased in USAAI Agents for Freight Brokers & 3PLs: Automate Quoting, Load Building & Customer ServiceDebales Works Across Email, Chat, SMS & WhatsApp. Automate Customer Communication,Process Orders 40% Faster, Save 5+ Hours a day & Scale Without Hiring.See It in ActionTrusted by leading Logistics Teamsand more...Every day starts here: Another routine ETA request lands in the inbox.Preparing your experiencePlayResetScene: 0Trusted by leadingLogistics TeamsGAIABAYGAIABAYGAIABAYMeet Your AI WorkforceAI Agents thatread & reply to emailsFour specialized AI agents. One shared brain. They know your customers, your shipments, and your workflow across every channel, 24/7.Email AI AgentSupport AI AgentSMS AI AgentPhone AI Agent& many more AI agentsorders@acmelogistics.com InboxKTKnight TransportDelay on Shipment #48219 · Carrier ~30min lateEx

In [5]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    text = re.sub(r'[^a-zA-Z0-9.,:%\- ]', '', text)  # remove junk chars
    return text

cleaned_text = clean_text(text)

In [6]:
cleaned_text

'Debales AI - Autonomous Logistics Automation Platform  AI Agents for Supply Chain  Debales AISolutions IntegrationsAI AgentsBlogCase StudiesSign UpSign InBased in USAAI Agents for Freight Brokers  3PLs: Automate Quoting, Load Building  Customer ServiceDebales Works Across Email, Chat, SMS  WhatsApp. Automate Customer Communication,Process Orders 40% Faster, Save 5 Hours a day  Scale Without Hiring.See It in ActionTrusted by leading Logistics Teamsand more...Every day starts here: Another routine ETA request lands in the inbox.Preparing your experiencePlayResetScene: 0Trusted by leadingLogistics TeamsGAIABAYGAIABAYGAIABAYMeet Your AI WorkforceAI Agents thatread  reply to emailsFour specialized AI agents. One shared brain. They know your customers, your shipments, and your workflow across every channel, 247.Email AI AgentSupport AI AgentSMS AI AgentPhone AI Agent many more AI agentsordersacmelogistics.com InboxKTKnight TransportDelay on Shipment 48219  Carrier 30min lateExceptionFrom: d

In [7]:
!pip install langchain

In [8]:
pip install langchain langchain-community langchain-core langchain-text-splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

docs = splitter.create_documents([cleaned_text])

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env file

api_key = os.getenv("api_key")


In [11]:
pip install sentence-transformers 

Note: you may need to restart the kernel to use updated packages.


In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def embed_text(cleaned_text):
    return model.encode(cleaned_text).tolist()

In [14]:
pip install pinecone

In [17]:
from pinecone import Pinecone, ServerlessSpec  # Added ServerlessSpec

pc = Pinecone(api_key=api_key)

index_name = "debales-ai"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'  # This is the most common region for free tier
        )
    )

index = pc.Index(index_name)


In [18]:
vectors = []

for i, doc in enumerate(docs):
    embedding = model.encode(doc.page_content).tolist()
    
    vectors.append({
        "id": str(i),
        "values": embedding,
        "metadata": {"text": doc.page_content}
    })

index.upsert(vectors)

UpsertResponse(upserted_count=43, _response_info={'raw_headers': {'date': 'Mon, 06 Apr 2026 22:56:32 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '83685', 'x-pinecone-request-latency-ms': '2791', 'x-envoy-upstream-service-time': '312', 'x-pinecone-response-duration-ms': '2793', 'grpc-status': '0', 'server': 'envoy'}})

In [57]:
def rag_search(query):
    query_embedding = model.encode(query).tolist()

    results = index.query(
        vector=query_embedding,
        top_k=8,
        include_metadata=True
    )

    context = "\n".join([
        match["metadata"]["text"]
        for match in results["matches"]
    ])

    score = results["matches"][0]["score"] if results["matches"] else 0

    return context, score

In [24]:
!pip install google-search-results

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32083 sha256=30dee89ab6b449336bac60a3b09e8aac0778ac3f84727314c10be0d10911e3d2
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\0c\47\f5\89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [ ]:
from serpapi import GoogleSearch


SERP_API_KEY = "put key"

def serp_search(query):
    params = {
        "engine": "google",
        "q": query,
        "api_key": SERP_API_KEY,
        "num": 5
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    output = []

    for r in results.get("organic_results", []):
        title = r.get("title", "")
        snippet = r.get("snippet", "")

        if snippet:
            output.append(f"{title}: {snippet}")

    return "\n".join(output) if output else "No results found"

In [64]:
def route_query(query):
    context, score = rag_search(query)

    query_lower = query.lower()

    debales_keywords = ["debales", "logistics", "ai agent", "supply chain"]

    is_debales = any(k in query_lower for k in debales_keywords)
    
    if is_debales and ("compare" in query_lower or "difference" in query_lower or "vs" in query_lower):
        return "both", context

    # 🔥 Case 1: Debales + decent match → RAG
    if is_debales and score > 0.5:
        return "rag", context

    # 🔥 Case 2: Debales but weak match → BOTH
    if is_debales:
        return "both", context

    # 🔥 Case 3: Not Debales → SERP
    return "serp", None

In [32]:
!pip install groq

In [ ]:
from groq import Groq
GROQ_API_KEY = "put key"
client = Groq(api_key=GROQ_API_KEY)



In [54]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

In [65]:
def generate_answer(query):
    route, context = route_query(query)

    if route == "rag":
        prompt = f"""
You are a strict AI assistant.

From the context, extract the MAIN purpose and key features.

Ignore:
- testimonials
- repeated content
- unrelated examples

Context:
{context}

Question: {query}

Answer in 2-3 lines maximum.
"""

    elif route == "serp":
        web_data = serp_search(query)

        prompt = f"""
        Answer using this web data:

        {web_data}

        Question: {query}
        """

    else:  # BOTH
        web_data = serp_search(query)

        prompt = f"""
        Use BOTH sources:

        Context:
        {context}

        Web Data:
        {web_data}

        Question: {query}
        """

    answer = call_llm(prompt)

    return {
        "route": route,
        "answer": answer
    }

In [ ]:
while True:
    query = input("Ask: ")

    result = generate_answer(query)

    print("\nRoute:", result["route"])
    print("Answer:", result["answer"])
